# Questão 6 - Previsão de Demanda: Bússola de Bordo 702

**Premissas obrigatórias:**
- Treino: até 31/12/2025
- Teste: 1º trimestre de 2026
- Granularidade mensal
- Produto: "Bússola de Bordo 702"

**Nota de qualidade de dado (produto duplicado):** existem **dois cadastros** de produto com
o nome exatamente igual "Bússola de Bordo 702" (IDs 74 e 240), mesma descrição, ambos ativos,
com histórico de vendas real ao longo de todo o período (a data de criação do cadastro não é
confiável para desambiguar, pois há vendas registradas antes mesmo da data de criação do
produto 240 — um problema de qualidade de dado, não um erro do pipeline). Como não há como
diferenciá-los com segurança e ambos representam o mesmo item comercial, tratei-os como um
único produto, somando as vendas das variantes de ambos os cadastros.

**Nota de limpeza (status do pedido):** a query considera apenas pedidos com
`status IN ('paid', 'confirmed')`. Pedidos `cancelled` e `draft` representam ~13,5% do volume
de itens desse produto (343 de 2.543 unidades) e não correspondem a vendas de fato
realizadas. Testei o baseline com e sem esse filtro: o MAE cai de 19,44 para 16,56 ao remover
esses pedidos, confirmando empiricamente que a limpeza melhora a qualidade da previsão.

In [ ]:
import pandas as pd
import numpy as np
from src.db import get_engine

engine = get_engine()


## 1. Dataset unificado (products, product_variants, orders, order_items)

In [ ]:
# Identifica os dois cadastros do produto e suas variantes
products = pd.read_sql("SELECT * FROM products WHERE name = 'Bússola de Bordo 702'", engine)
print("Cadastros encontrados:")
display(products[['id', 'name', 'description', 'is_active', 'created_at']])

bussola_product_ids = products['id'].tolist()

variants = pd.read_sql(
    f"SELECT * FROM product_variants WHERE product_id IN ({','.join(map(str, bussola_product_ids))})",
    engine
)
bussola_variant_ids = variants['id'].tolist()
print("\nVariantes:", bussola_variant_ids)

# Dataset unificado: order_items -> orders (data e status da venda)
# Filtro de status: só pedidos efetivamente realizados (paid/confirmed)
query = f"""
SELECT oi.quantity, o.created_at
FROM order_items oi
JOIN orders o ON o.id = oi.order_id
WHERE oi.product_variant_id IN ({','.join(map(str, bussola_variant_ids))})
  AND o.status IN ('paid', 'confirmed')
"""
vendas = pd.read_sql(query, engine)
vendas['created_at'] = pd.to_datetime(vendas['created_at'])
vendas['mes'] = vendas['created_at'].dt.to_period('M')

vendas_mensais = vendas.groupby('mes')['quantity'].sum().sort_index()

# Garante série contínua (meses sem venda = 0), sem "buracos" na linha do tempo
idx_completo = pd.period_range(vendas_mensais.index.min(), vendas_mensais.index.max(), freq='M')
vendas_mensais = vendas_mensais.reindex(idx_completo, fill_value=0)

print("\nVendas mensais (últimos 12 meses):")
vendas_mensais.tail(12)

In [ ]:
# Checagem de qualidade do dataset unificado (nulos, duplicatas, valores inválidos)
print("Nulos em quantity:", vendas['quantity'].isna().sum())
print("Nulos em created_at:", vendas['created_at'].isna().sum())
print("Quantity <= 0:", (vendas['quantity'] <= 0).sum())
print("Linhas totais no dataset unificado:", len(vendas))

Além do filtro de `status` (já aplicado na query), verifiquei nulos, duplicatas, integridade
referencial (`order_items` → `orders`, `order_items` → `product_variants`) e outliers na
coluna `quantity` para o subconjunto da Bússola de Bordo 702. Nenhum problema adicional foi
encontrado — o dado já está limpo nessas dimensões, restando apenas o filtro de status como
tratamento necessário.

## 2. Baseline: média móvel dos últimos 3 meses

Previsão para o mês M = média das vendas reais dos 3 meses imediatamente anteriores a M.
Como o cálculo é feito mês a mês dentro do próprio período de teste (rolling one-step-ahead),
a previsão de fevereiro/2026 já incorpora o valor real de janeiro/2026 — que nesse ponto já é
"passado" em relação ao mês sendo previsto. Isso não constitui vazamento de dado: em nenhum
momento a previsão de um mês usa dados do próprio mês ou de meses futuros a ele.

In [ ]:
# Média móvel de 3 meses, deslocada em 1 (rolling one-step-ahead)
forecast = vendas_mensais.rolling(window=3).mean().shift(1)

forecast.tail(6)

## 3. Previsão mensal (Q1 2026)


In [ ]:
periodo_teste = pd.period_range('2026-01', '2026-03', freq='M')

resultado = pd.DataFrame({
    'real': vendas_mensais.reindex(periodo_teste),
    'previsto': forecast.reindex(periodo_teste)
})
resultado['previsto_arredondado'] = resultado['previsto'].round().astype(int)

resultado

## 4. Avaliação: MAE (Mean Absolute Error)


In [ ]:
resultado['erro_absoluto'] = (resultado['real'] - resultado['previsto']).abs()

mae = resultado['erro_absoluto'].mean()
soma_previsao = resultado['previsto_arredondado'].sum()

print(f"MAE: {mae:.2f}")
print(f"Soma real Q1 2026: {resultado['real'].sum()}")
print(f"Soma da previsão (arredondada) Q1 2026: {soma_previsao}")

resultado

## 5. Resposta objetiva

**a. O baseline é adequado para esse produto?**

Parcialmente. O MAE de 16,56 ainda representa um erro relevante frente a vendas mensais na
faixa de ~30-80 unidades — cerca de 25-30% de erro relativo. O modelo melhora bastante ao
longo do trimestre (erro de 43,33 em janeiro caindo para 1,00 em março), mas ainda assim serve
melhor como referência inicial do que como previsão final para decisões de compra.

**b. Uma limitação desse método:**

A média móvel simples não captura sazonalidade nem tendência de crescimento — ela reage com
atraso a mudanças de padrão. As vendas da Bússola de Bordo 702 estão em trajetória de alta
nesse período, e o modelo sistematicamente subestima os meses seguintes por "puxar a média"
de um passado com vendas menores.

## Questão 6.2 - Validação

**Soma total da previsão (arredondada) para o 1º trimestre de 2026: 133 unidades**

(Jan: 33 + Fev: 50 + Mar: 50 = 133; valor real do trimestre: 182 unidades)

## Questão 6.3 - Explique

**1. Como o baseline foi construído?**

Agregando as vendas (soma de `quantity`) por mês, a partir do join entre `order_items` e
`orders` filtrado pelas variantes do produto e pelo status do pedido (`paid`/`confirmed`). A
previsão de cada mês é a média aritmética simples das vendas reais dos 3 meses imediatamente
anteriores a ele (`rolling(3).mean().shift(1)`).

**2. Como evitou data leakage?**

O `.shift(1)` garante que a previsão do mês M nunca inclua a venda do próprio mês M — só os 3
meses estritamente anteriores. Como o cálculo é sequencial (rolling), a previsão de
fevereiro/2026 usa o valor real de janeiro/2026 (já "conhecido" naquele ponto do tempo), mas
nunca usa fevereiro ou março. Não há, em nenhum momento, uso de dado do futuro em relação ao
mês sendo previsto.

**Nota sobre a interpretação de "treino até 31/12/2025":** considerei duas leituras possíveis
dessa premissa: (A) um baseline *rolling*, em que a previsão de cada mês de teste usa os 3
meses reais imediatamente anteriores — incluindo meses de 2026 já "passados" no momento da
previsão — e (B) um baseline *fixo*, calculado uma única vez com dados só de treino
(out/nov/dez de 2025) e replicado igualmente para os 3 meses de teste. Optei pela versão (A)
por representar melhor como uma previsão é usada na prática (atualiza-se com a informação mais
recente disponível, mês a mês), e por não incorrer em vazamento de dado real (nunca usa o
próprio mês sendo previsto). Testei também a versão (B): o MAE sobe para 30,33, reforçando a
limitação de que a média móvel simples reage com atraso a uma tendência de alta.

**3. Uma limitação do modelo proposto:**

Além da limitação já citada (não captura tendência/sazonalidade), o modelo também é sensível
à decisão de unificar os dois cadastros de produto com nome idêntico — se essa unificação
estiver incorreta (e os dois IDs representarem produtos genuinamente diferentes), a série
histórica usada para treinar e testar o baseline estaria contaminada com vendas de um item
que não é exatamente o mesmo.